# Эксперименты и результаты

Ноутбук использует пакет `deepfake` — здесь только запуск и разбор, вся логика
живёт в `src/deepfake/`. То же самое из терминала:

```bash
python scripts/train.py --all
python scripts/report.py
```

Все запуски идут через `run_experiment()`, поэтому подчиняются единому
протоколу: свежие веса, один и тот же split, одинаковый бюджет обучения, порог и
чекпоинт по валидации, метрики по тесту.

In [ ]:
import sys
from pathlib import Path

# пакет лежит в src/ — добавляем в путь, если проект не установлен через pip install -e .
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt

from deepfake.config import CLEAN_DATASET, COMPARISON_CSV, DEFAULT_TRAINING, NOISY_DATASET, get_device
from deepfake.experiment import run_experiment
from deepfake.metrics import REGISTRY
from deepfake.models import MODEL_FACTORIES, build_model, count_parameters

plt.rcParams["figure.dpi"] = 110

print("устройство:", get_device())
print("бюджет обучения:", DEFAULT_TRAINING)

In [ ]:
for name in MODEL_FACTORIES:
    print(f"{name:22s} {count_parameters(build_model(name)) / 1e6:7.2f}M параметров")

## Сравнение архитектур

Все на очищенных данных, всё остальное одинаково. Разница в таблице объясняется
только архитектурой.

In [ ]:
CONFIG = DEFAULT_TRAINING         # .scaled(epochs=3) для быстрой проверки

trained = {}
for name in MODEL_FACTORIES:
    model, history, result = run_experiment(name, CLEAN_DATASET, note="очищенные данные", config=CONFIG)
    trained[name] = (model, history)

## Кривые обучения

In [ ]:
def plot_history(history, title):
    epochs = range(1, len(history) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(epochs, history.train_loss, label="train")
    axes[0].plot(epochs, history.val_loss, label="val")
    axes[0].set_yscale("log")
    axes[0].set_title("Loss")

    axes[1].plot(epochs, history.val_f1, label="val @ 0.5")
    axes[1].plot(epochs, history.val_tuned_f1, "--", label="val @ подобранный порог")
    axes[1].set_title("F1")

    axes[2].plot(epochs, history.val_roc_auc)
    axes[2].set_title("ROC-AUC (val)")

    for axis in axes:
        axis.set_xlabel("эпоха")
        axis.grid(alpha=0.3)
    axes[0].legend()
    axes[1].legend()

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


for name, (_, history) in trained.items():
    plot_history(history, name)

Разрыв между сплошной и пунктирной линией на графике F1 — это то, что даёт
подбор порога. При дисбалансе 5:1 он обычно заметный.

## Вклад очистки от шума

Та же модель на шумной версии того же разбиения. Изображения и split совпадают,
отличается только предобработка.

In [ ]:
run_experiment(
    "InceptionV1", NOISY_DATASET,
    label="InceptionV1 (шумные)", note="без удаления шума", config=CONFIG,
)

## Шум как предобучение

Сначала шумные данные, затем очищенные. Утечки нет: hold-out обеих версий
состоит из одних и тех же id, поэтому первая стадия не видит тестовых
изображений второй.

In [ ]:
stage_one, _, _ = run_experiment(
    "InceptionV1", NOISY_DATASET,
    label="InceptionV1 (стадия 1)", config=CONFIG,
)
REGISTRY.results.pop("InceptionV1 (стадия 1)", None)   # промежуточный шаг в таблицу не идёт

run_experiment(
    "InceptionV1", CLEAN_DATASET,
    label="InceptionV1 (шум -> чистые)", note="двухстадийное обучение",
    config=CONFIG, model=stage_one,
)

## Итоговая таблица

In [ ]:
table = REGISTRY.to_frame()
REGISTRY.save(COMPARISON_CSV)
table

In [ ]:
ordered = table.sort_values("f1")

fig, axis = plt.subplots(figsize=(9, 0.45 * len(ordered) + 2))
axis.barh(ordered.index, ordered["f1"], color="#4C72B0")
axis.set_xlim(max(0.0, ordered["f1"].min() - 0.05), 1.0)
axis.set_xlabel("F1 на тестовой выборке")
axis.grid(axis="x", alpha=0.3)
axis.set_title("Сравнение архитектур")

for index, value in enumerate(ordered["f1"]):
    axis.text(value, index, f" {value:.4f}", va="center", fontsize=9)

fig.tight_layout()
plt.show()

## Выводы

Итоговый отчёт с автоматически посчитанными разницами по каждой паре
экспериментов собирается командой:

```bash
python scripts/report.py
```

Он перезапишет `RESULTS.md` — цифры там берутся из `results/model_comparison.csv`
и не могут разойтись с посчитанными здесь.